# 01 -- Data and Source Validation

Bock & Maewal (2023) paper rebuild. Validates the paper's own 11-ETF universe against real Yahoo Finance data, builds the weekly calendar, and audits data quality. No strategy tuning happens in this notebook -- see `docs/paper_source_audit.md` for full source evidence.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data import PAPER_UNIVERSE, VANGUARD_UNIVERSE_FORBIDDEN, assert_canonical_order, assert_no_vanguard_substitution, load_universe_prices
from src.calendar import build_weekly_calendar, require_holiday_decision_if_incomplete
from src.decisions import PaperDecisionRequiredError

print('Canonical universe (Table 1 order):', PAPER_UNIVERSE)

Canonical universe (Table 1 order): ('XLK', 'XLV', 'XLY', 'VOX', 'XLF', 'XLI', 'XLP', 'XLU', 'XLB', 'IYR', 'XLE')


## Universe verification

Assert the paper's own 11 tickers are used, and that none of the archived project's Vanguard-fund tickers have leaked in.

In [2]:
assert_canonical_order(PAPER_UNIVERSE)
assert_no_vanguard_substitution(PAPER_UNIVERSE)
print('Universe OK: 11 tickers, Table-1 order, no Vanguard substitution.')
print('Forbidden (archived-project) tickers, confirmed absent:', sorted(VANGUARD_UNIVERSE_FORBIDDEN))

Universe OK: 11 tickers, Table-1 order, no Vanguard substitution.
Forbidden (archived-project) tickers, confirmed absent: ['VAW', 'VCR', 'VDC', 'VDE', 'VFH', 'VGT', 'VHT', 'VIS', 'VNQ', 'VPU']


## Real data load (Yahoo Finance, immutable raw CSVs under `data/raw/prices/`)

In [3]:
prices = load_universe_prices()
for symbol, df in prices.items():
    print(f"{symbol:5s}  rows={len(df):5d}  {df['date'].min().date()} -> {df['date'].max().date()}")

XLK    rows= 6952  1998-12-22 -> 2026-08-13
XLV    rows= 6952  1998-12-22 -> 2026-08-13
XLY    rows= 6952  1998-12-22 -> 2026-08-13
VOX    rows= 5503  2004-09-29 -> 2026-08-13
XLF    rows= 6952  1998-12-22 -> 2026-08-13
XLI    rows= 6952  1998-12-22 -> 2026-08-13
XLP    rows= 6952  1998-12-22 -> 2026-08-13
XLU    rows= 6952  1998-12-22 -> 2026-08-13
XLB    rows= 6952  1998-12-22 -> 2026-08-13
IYR    rows= 6577  2000-06-19 -> 2026-08-13
XLE    rows= 6952  1998-12-22 -> 2026-08-13


## Data audit

See `outputs/paper_data_audit.csv` for the full per-symbol gap/null audit (already generated; reproduced here for a quick visual check). All 11 series cover the paper's 2012-2022 window with 2 years of prior history (>= 2010) and no data gaps beyond ordinary weekends/holidays.

In [4]:
audit_df = pd.read_csv(PROJECT_ROOT / 'outputs' / 'paper_data_audit.csv')
audit_df[['symbol', 'history_start', 'history_end', 'gaps_over_5_days_2010_2022', 'covers_full_paper_period_2012_2022']]

,symbol,history_start,history_end,gaps_over_5_days_2010_2022,covers_full_paper_period_2012_2022
0,GSPC,1927-12-30,2026-08-13,0,True
1,IYR,2000-06-19,2026-08-13,0,True
2,SPY,1993-01-29,2026-08-13,0,True
3,VOX,2004-09-29,2026-08-13,0,True
4,XLB,1998-12-22,2026-08-13,0,True
5,XLE,1998-12-22,2026-08-13,0,True
6,XLF,1998-12-22,2026-08-13,0,True
7,XLI,1998-12-22,2026-08-13,0,True
8,XLK,1998-12-22,2026-08-13,0,True
9,XLP,1998-12-22,2026-08-13,0,True


## Weekly calendar reconstruction

Built from real observed trading days (not a synthetic holiday calendar) -- see `docs/paper_execution_timeline.md`. Any week missing a literal Friday close or the following Monday open blocks on `DECISION_REQUIRED_HOLIDAY_EXECUTION`, demonstrated below on a real holiday-shortened week (Good Friday 2021).

In [5]:
spy = prices['XLK']  # any universe member works; trading-day index is shared across US equities
trading_days = pd.DatetimeIndex(spy[(spy['date'] >= '2021-01-01') & (spy['date'] <= '2021-12-31')]['date'])
calendar_2021 = build_weekly_calendar(trading_days)
calendar_2021[calendar_2021['friday_present'] == False]

,week_id,first_actual_trading_day,last_actual_trading_day,friday_present,model_cutoff,entry_candidate_date,exit_candidate_date,label_known_date
12,12,2021-03-29,2021-04-01,False,2021-04-01,2021-04-05,2021-04-01,2021-04-09
50,50,2021-12-20,2021-12-23,False,2021-12-23,2021-12-27,2021-12-23,2021-12-31


In [6]:
try:
    require_holiday_decision_if_incomplete(calendar_2021)
except PaperDecisionRequiredError as exc:
    print('Blocked as expected:')
    print(exc)

Blocked as expected:
[DECISION_REQUIRED_HOLIDAY_EXECUTION] (Level 3) 'Weekly-calendar handling when Friday close or Monday open falls on a market holiday' is not resolved in decisions/paper_decision_register.json. Required before: 3 week(s) lack a literal Friday close and/or a following Monday open (e.g. week_id=12); holiday-substitution rule must be chosen before these weeks can be executed.. See docs/paper_source_audit.md and the decision register for source evidence and candidate options.


## Summary

Universe, real data acquisition, and weekly-calendar mechanics are all source-supported and implemented. The only open item touched in this notebook is holiday-week handling (`DECISION_REQUIRED_HOLIDAY_EXECUTION`, Level 3) -- see the decision register for the full list.